In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending (usually this block will be the actual code block that involves sending emails or API calls)
    return f"Email sent"

To ensure human-in-the-loop process, you can add the prebuilt `HumanInTheLoopMiddleware` function with several arguments to specify (i.e. which tools require HITL, which do not)

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True, # Require human approval before any email is sent out
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem, '
                                                                          "let's "
                                                                          'reschedule. '
                                                                          'What '
                                                                          'time '
                                                                          'works '
                                                                          'best '
                                                                          'for '
                                                                          'you? '
       

- Notice that we now have a new object within the response: interrupt
- Also notice that the final message is an AIMessage with empty content
- We have interrupted/stopped the agent at the point that the agent has made the tool call

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, no problem, let's reschedule. What time works best for you? Best, Seán."}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, no problem, let\'s reschedule. What time works best for you? Best, Seán."}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='26cc46356fd16f78dc42f2a41bfa580d', response_schema=None)]


In [7]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem, let's reschedule. What time works best for you? Best, Seán.


## Approve

In [8]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]} # To approve the agent suggestions, just invoke the agent again with our decision
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='bd4cf636-ace8-4755-893e-de69d4a95a51'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "inbox"}'}, '__gemini_function_call_thought_signatures__': {'call_84309': 'EnMKcQFpFH0TzryqXzWJKK9DMX0QlNXgDsCw8vbyWdQ8D/krVxc+B1DVN+l3UAkNe0o9/i/G8+vhiDq4yujPRxHxemSXYa9zOzYxs7y67/+oqCEOnvQnJaxtGKuGflie82D6k8FXvD05rA6bloR3PBhTcLdP'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f6e7-961b-79d0-9bd3-8a0146010d8a-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'inbox'}, 'id': 'call_84309', 'type': 'tool_call'}], invalid_to

## Reject

In [15]:
# Refresh the conversation with a new thread, since the previous conversation thread (ID 1) was already approved

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "3"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

from pprint import pprint

pprint(response)

# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                    

In [16]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                    

- In this example, we rejected the sending of the email because the sign off was not what we wanted. We wanted it to be "Your merciful leader, Sean". Therefore we reject the tool call from executing and instruct it to sign off with that ending.
- It is another interrupt because now we encounter the need for human approval. If we want to continue sending it, we need to follow up by invoking the agent with a decision to approve the email.

In [17]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem at all. Let me know what time works for you to reschedule.

Your merciful leader, Seán.


## Edit

In [18]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='96f9bd51-7dd9-4e7d-bb8a-74a1175f9a9d'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "me"}'}, '__gemini_function_call_thought_signatures__': {'call_277025': 'EnMKcQFpFH0TuUjxCn2b6t0TNvDI8XME6dwyvkmuj6CyMetNWqk/8ABs/eMr+8P8/U1E+69h73lKzlvcWt9YSWg24npPfPtJJ3UzAZyvc3+V7rKQLqwBOtIx/RV+uUU5gcn6Mtbi9UQHrXG+FMOvMztztw0z'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f6f7-17cd-7bb2-8d50-efd915baa909-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'me'}, 'id': 'call_277025', 'type': 'tool_call'}], invalid_tool_c

- Notice that the tool response in the last ToolMessage indicates "Email sent".
- Therefore, the edit decision makes the new tool call and will not loop back again to seek approval.